In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, jaccard_score, precision_score, recall_score, f1_score, accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Import utilities from the other files
from data_utils_seg import SegmentationDataset, CLASS_NAMES
from model_utils_seg import get_model, combined_loss, iou_score

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8 # SegFormer can be memory intensive, adjust if needed
NUM_EPOCHS = 50
LR = 1e-4
IMAGE_SIZE = 128
NUM_CLASSES = 7 # We are back to 7-class semantic segmentation

# --- Data Path Placeholders ---
BASE_PATH = "../Dataset/Dataset/Prepared_Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_PATH, "train/masks")
VAL_IMG_DIR = os.path.join(BASE_PATH, "val/images")
VAL_MASK_DIR = os.path.join(BASE_PATH, "val/masks")

# --- Augmentations ---
train_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.RandomRotate90(), A.HorizontalFlip(), A.VerticalFlip(),
    A.Affine(rotate=(-15, 15), scale=(0.9, 1.1), translate_percent=(0.06, 0.06)),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# --- Evaluation Function ---
def validate_epoch(model, val_loader):
    model.eval()
    val_loss_sum, val_miou_sum = 0.0, 0.0
    all_preds, all_masks = [], []
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE).squeeze(-1).long()
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = combined_loss(logits, masks)
            
            val_loss_sum += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            val_miou_sum += iou_score(logits, masks, num_classes=NUM_CLASSES) * imgs.size(0)
            all_preds.append(preds.cpu().numpy())
            all_masks.append(masks.cpu().numpy())

    avg_loss = val_loss_sum / len(val_loader.dataset)
    avg_miou = val_miou_sum / len(val_loader.dataset)
    return avg_loss, avg_miou, np.concatenate(all_preds, axis=0), np.concatenate(all_masks, axis=0)

# --- Plotting Function ---
def plot_results(train_losses, val_losses, val_mious, all_val_preds, all_val_masks, val_dataset, session_title):
    print(f"\n--- Generating Visualizations for: {session_title} ---")
    
    plt.figure(figsize=(14, 6)); plt.suptitle(session_title, fontsize=16)
    plt.subplot(1, 2, 1); plt.plot(train_losses, label='Train Loss'); plt.plot(val_losses, label='Validation Loss'); plt.title('Training & Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2); plt.plot(val_mious, label='Validation mIoU', color='orange'); plt.title('Validation Mean IoU (mIoU)'); plt.xlabel('Epoch'); plt.ylabel('mIoU'); plt.legend(); plt.grid(True)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.show(block=False)
    
    flat_true, flat_pred = all_val_masks.flatten(), all_val_preds.flatten()
    cm = confusion_matrix(flat_true, flat_pred, labels=np.arange(NUM_CLASSES))
    plt.figure(figsize=(10, 8)); sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES); plt.xlabel('Predicted Label'); plt.ylabel('True Label'); plt.title(f'Confusion Matrix - {session_title}'); plt.show(block=True)

# --- Training Session Function ---
def run_training_session(model_name, encoder_name, enhancement_mode, model_save_path):
    print("\n" + "="*70 + f"\nSTARTING TRAINING: Model='{model_name.upper()}', Encoder='{encoder_name}', Enhancement='{enhancement_mode}'\n" + "="*70 + "\n")

    train_dataset = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform, use_enhancement=enhancement_mode)
    val_dataset = SegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=val_transform, use_enhancement='none')
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    model = get_model(model_name, DEVICE, encoder_name=encoder_name, classes=NUM_CLASSES)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, mode='max')
    scaler = torch.cuda.amp.GradScaler()

    best_miou = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_miou': []}
    final_val_preds, final_val_masks = None, None

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} (Train)")
        for imgs, masks in progress_bar:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE).squeeze(-1).long()
            optimizer.zero_grad();
            with torch.cuda.amp.autocast(): loss = combined_loss(model(imgs), masks)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            train_loss += loss.item() * imgs.size(0)
            progress_bar.set_postfix(loss=loss.item())

        history['train_loss'].append(train_loss / len(train_loader.dataset))
        val_loss, val_miou, current_preds, current_masks = validate_epoch(model, val_loader)
        history['val_loss'].append(val_loss); history['val_miou'].append(val_miou); scheduler.step(val_miou)
        print(f"Epoch {epoch} finished. Train Loss: {history['train_loss'][-1]:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(model.state_dict(), model_save_path)
            print(f"Model saved to {model_save_path}! (Best mIoU: {best_miou:.4f})")
            final_val_preds, final_val_masks = current_preds, current_masks

    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path)); _, _, final_val_preds, final_val_masks = validate_epoch(model, val_loader)
    plot_results(history['train_loss'], history['val_loss'], history['val_miou'], final_val_preds, final_val_masks, val_dataset, session_title=f"Model: {model_name.upper()} ({encoder_name}) | Enhancement: {enhancement_mode}")

# --- Main Execution ---
def main():
    if not os.path.exists(TRAIN_IMG_DIR) or not os.path.exists(VAL_IMG_DIR):
        print(f"!! ERROR: Data paths incorrect. Expected Prepared_Dataset directory at: {BASE_PATH}")
        return

    # --- SegFormer with MiT-B5 Backbone Training Sessions ---
    model = 'Segformer'
    encoder = 'mit_b5'
    
    print(f"--- Starting Training for {model.upper()} with {encoder.upper()} Backbone ---")
    
    run_training_session(model, encoder, 'none', f"{model.lower()}_{encoder}_none_50.pth")
    run_training_session(model, encoder, 'all', f"{model.lower()}_{encoder}_preprocess_50.pth")
    run_training_session(model, encoder, 'hybrid', f"{model.lower()}_{encoder}_hybrid_50.pth")
    
    print("\nAll requested training sessions completed.")

if __name__ == '__main__':
    try:
        main()
    except Exception as e:
        print(f"\nAn unexpected and fatal error occurred: {e}")

c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Starting Training for SEGFORMER with MIT_B5 Backbone ---

STARTING TRAINING: Model='SEGFORMER', Encoder='mit_b5', Enhancement='none'

Initializing SEGFORMER model with mit_b5 encoder.


c:\Users\veerk\OneDrive\Desktop\DIP Project\dip\Lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\veerk\.cache\huggingface\hub\models--smp-hub--mit_b5.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\veerk\AppData\Local\Temp\ipykernel_9856\3428248048.py:96: FutureWarning: `torch.cuda.amp

KeyboardInterrupt: 